In [2]:

import pandas as pd
import requests
from pathlib import Path
import time  # Pour les délais

# Configuration
BASE_URL = "http://localhost:8080/api/products"
TAUX_CHANGE = 0.04  # Conversion du prix (₹ vers autre devise)

# Charger le CSV
df = pd.read_csv('amazon.csv')

# Conversion des prix
df['actual_price'] = (
    df['actual_price']
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float) * TAUX_CHANGE
).round(2)

# Renommer les colonnes pour correspondre au modèle de données du backend
df.rename(columns={
    'product_name': 'name',
    'about_product': 'description',
    'img_link': 'imageUrl',
    'actual_price': 'price'
}, inplace=True)

# Téléchargement des images (optionnel si vous voulez stocker localement)
def download_image(url, filename):
    try:
        response = requests.get(url, timeout=20)
        if response.status_code == 200:
            save_path = Path(f"src/main/resources/static/images/{filename}")
            save_path.parent.mkdir(parents=True, exist_ok=True)
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return f"/images/{filename}"
    except Exception as e:
        print(f" Image non téléchargée : {e}")
    return None

# Télécharger les images et mettre à jour la colonne imageUrl
df['imageUrl'] = [
    download_image(row['imageUrl'], f"product_{index+1}.jpg")
    for index, row in df.iterrows()
]

# Envoyer les données au backend
for _, row in df.iterrows():
    try:
        # Construire le payload correctement
        data = {
            "name": row['name'],
            "description": row['description'],
            "price": row['price'],
            "stock": 10  # Stock initial par défaut (vous pouvez ajuster selon vos besoins)
        }

        # Gérer l'image (si elle existe)
        if row['imageUrl']:
            image_path = f"src/main/resources/static{row['imageUrl']}"
            with open(image_path, 'rb') as img_file:
                files = {'image': (row['imageUrl'].split('/')[-1], img_file, 'image/jpeg')}
                response = requests.post(
                    BASE_URL,
                    data=data,
                    files=files,
                    timeout=30
                )
        else:
            # Si pas d'image, envoyer uniquement les données
            response = requests.post(BASE_URL, json=data, timeout=30)

        if response.ok:
            print(f" {row['name']} ajouté")
        else:
            print(f" Échec : {response.text}")

        # Délai pour éviter les erreurs de surcharge
        time.sleep(1)

    except Exception as e:
        print(f" Erreur critique : {e}")

 Échec : {"timestamp":"2025-03-02T14:36:12.249+00:00","status":400,"error":"Bad Request","trace":"org.springframework.web.bind.MissingServletRequestParameterException: Required request parameter 'name' for method parameter type String is not present\r\n\tat org.springframework.web.method.annotation.RequestParamMethodArgumentResolver.handleMissingValueInternal(RequestParamMethodArgumentResolver.java:220)\r\n\tat org.springframework.web.method.annotation.RequestParamMethodArgumentResolver.handleMissingValue(RequestParamMethodArgumentResolver.java:196)\r\n\tat org.springframework.web.method.annotation.AbstractNamedValueMethodArgumentResolver.resolveArgument(AbstractNamedValueMethodArgumentResolver.java:125)\r\n\tat org.springframework.web.method.support.HandlerMethodArgumentResolverComposite.resolveArgument(HandlerMethodArgumentResolverComposite.java:122)\r\n\tat org.springframework.web.method.support.InvocableHandlerMethod.getMethodArgumentValues(InvocableHandlerMethod.java:226)\r\n\tat 